# Aerial Building Segmentation — Results & Visualisation

This notebook walks through the full experiment pipeline:
1. Setup repo and dependencies
2. Download the INRIA dataset automatically via Kaggle
3. Explore the dataset
4. Train UNet (15 epochs, 50k tiles, ~45–60 min on A100)
5. Evaluate on held-out West Tyrol validation set
6. Visualise qualitative predictions side-by-side
7. Plot learning curves

**Training scripts live in `src/` and are run with `!python`.  
This notebook is for exploration, monitoring, and visualisation.**

## 1. Setup

Clone the repo and install dependencies. Data and checkpoints stay on Colab's local disk (`/content/`) — no Drive needed.

In [ ]:
import os

REPO_DIR   = '/content/aerial-segmentation'
DATA_ROOT  = '/content/inria'
CKPT_BASE  = '/content/checkpoints'

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_BASE, exist_ok=True)
print('Paths ready. No Drive needed.')

In [ ]:
# Clone the correct branch (or pull if already present)
!git clone -b claude/focused-wilson \
    https://github.com/ibrahim-2337/aerial-segmentation.git \
    /content/aerial-segmentation 2>/dev/null \
 || git -C /content/aerial-segmentation pull origin claude/focused-wilson
%cd /content/aerial-segmentation

In [ ]:
# Install dependencies (repo already cloned and cd'd by cell above)
!pip install -q -r requirements.txt

import sys
if '/content/aerial-segmentation' not in sys.path:
    sys.path.insert(0, '/content/aerial-segmentation')

print('Setup complete.')

## 2. Dataset Download

Downloads the INRIA dataset directly to Colab's local disk (`/content/inria`). ~10–15 min on first run.  
No Drive involved — data lives only for this session (~200 GB free on Colab local disk).

In [ ]:
import os
import json

TRAIN_IMAGES = f'{DATA_ROOT}/train/images'

if os.path.isdir(TRAIN_IMAGES) and len(os.listdir(TRAIN_IMAGES)) > 0:
    print(f'Dataset already present ({len(os.listdir(TRAIN_IMAGES))} images). Skipping download.')
else:
    print('Downloading INRIA dataset to /content/inria ...')

    # Kaggle credentials from Colab Secrets
    from google.colab import userdata
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    creds = {'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')}
    creds_path = os.path.join(kaggle_dir, 'kaggle.json')
    with open(creds_path, 'w') as f:
        json.dump(creds, f)
    os.chmod(creds_path, 0o600)
    print(f'Kaggle user: {creds["username"]}')

    !pip install -q kaggle

    # Download and unzip directly to DATA_ROOT — no Drive involved
    !kaggle datasets download \
        -d sagar100rathod/inria-aerial-image-labeling-dataset \
        -p /content/inria_tmp --unzip --quiet

    # Move train/ into DATA_ROOT
    import shutil
    src_train = '/content/inria_tmp/AerialImageDataset/train'
    shutil.move(src_train, DATA_ROOT)
    shutil.rmtree('/content/inria_tmp', ignore_errors=True)

    print(f'Done. Images at {TRAIN_IMAGES}: {len(os.listdir(TRAIN_IMAGES))} files')

print(f'DATA_ROOT = {DATA_ROOT}')

## 3. Dataset Exploration

The INRIA dataset contains georeferenced GeoTIFF images at 30 cm/pixel resolution, each 5000 × 5000 px.  Ground-truth masks are binary: white (255) = building, black (0) = background.

Here we open one example with `rasterio` to verify the coordinate reference system (CRS) metadata is preserved, then display a tile sample to sanity-check the data pipeline.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

images_dir = Path(DATA_ROOT) / 'train' / 'images'
gt_dir     = Path(DATA_ROOT) / 'train' / 'gt'

tif_files = sorted(images_dir.glob('*.tif'))
print(f'Total training images: {len(tif_files)}')
for f in tif_files[:5]:
    print(f'  {f.name}')

In [ ]:
# Inspect coordinate reference system metadata
sample_img  = tif_files[0]
sample_mask = gt_dir / sample_img.name

with rasterio.open(sample_img) as src:
    print(f'Image : {sample_img.name}')
    print(f'  Size      : {src.width} x {src.height}')
    print(f'  Bands     : {src.count}')
    print(f'  CRS       : {src.crs}')
    print(f'  Transform : {src.transform}')

with rasterio.open(sample_mask) as src:
    mask_data = src.read(1)
    print(f'\nMask : {sample_mask.name}')
    print(f'  Unique values    : {np.unique(mask_data)}')
    print(f'  Building coverage: {(mask_data > 127).mean()*100:.1f}%')

In [ ]:
import rasterio.windows as rw

# Find a tile with meaningful building coverage rather than hardcoding coordinates
with rasterio.open(sample_mask) as src:
    mask_full = src.read(1)

found_row, found_col = 0, 0
for r in range(0, mask_full.shape[0] - 256, 256):
    for c in range(0, mask_full.shape[1] - 256, 256):
        if (mask_full[r:r+256, c:c+256] > 127).mean() > 0.08:
            found_row, found_col = r, c
            break
    else:
        continue
    break

window = rw.Window(found_col, found_row, 256, 256)
with rasterio.open(sample_img) as src:
    tile_img = src.read(window=window)
with rasterio.open(sample_mask) as src:
    tile_msk = src.read(1, window=window)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.moveaxis(tile_img, 0, -1))
axes[0].set_title('RGB Tile')
axes[1].imshow(tile_msk, cmap='gray')
axes[1].set_title('Ground Truth Mask')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'{sample_img.name}  —  tile @ ({found_row}, {found_col})', y=1.02)
plt.tight_layout()
plt.show()
print(f'Building pixels in tile: {(tile_msk > 127).mean()*100:.1f}%')

In [ ]:
from src.dataset import InriaDataset, TRAIN_CITIES, VAL_CITIES

# overlap=0.0 is consistent with training and builds the index fast
train_ds = InriaDataset(DATA_ROOT, split='train', tile_size=256, overlap=0.0)
val_ds   = InriaDataset(DATA_ROOT, split='val',   tile_size=256, overlap=0.0)

print(f'Train cities : {TRAIN_CITIES}')
print(f'Val   cities : {VAL_CITIES}')
print(f'Train tiles  : {len(train_ds):,}  (overlap=0.0)')
print(f'Val   tiles  : {len(val_ds):,}  (overlap=0.0)')

img_t, msk_t = train_ds[0]
print(f'\nImage tensor : {img_t.shape}  dtype={img_t.dtype}')
print(f'Mask  tensor : {msk_t.shape}  dtype={msk_t.dtype}')

## 4. Training

Trains UNet for 15 epochs using 50 000 randomly sampled tiles (no overlap) — about 23% of the full dataset.  
Estimated **~45–60 min on A100**. Expected val IoU: 0.65+.

To go deeper: set `--max_tiles 0` (all 219k tiles, ~3–4 hrs) and `--epochs 20`.

In [ ]:
# Scaled run: UNet, 15 epochs, 50 000 train tiles, 12 500 val tiles
# ~45-60 min on A100. Set --max_tiles 0 for the full 219k tile dataset.
!python src/train.py \
    --model unet \
    --seed 42 \
    --epochs 15 \
    --max_tiles 50000 \
    --overlap 0.0 \
    --data_root {DATA_ROOT}

## 5. Results — Comparison Table

After training we evaluate every checkpoint on the held-out West Tyrol validation set.  The evaluation script loads the best checkpoint for each model/seed, computes IoU and Dice over all validation tiles, and writes `experiments/results.csv`.

The table aggregates across the two seeds per model showing **mean ± std** to quantify run-to-run variance.

In [ ]:
!python src/evaluate.py --mode metrics --data_root {DATA_ROOT}

In [ ]:
import pandas as pd
import numpy as np

results_path = f'{REPO_DIR}/experiments/results.csv'
try:
    df = pd.read_csv(results_path).dropna(subset=['iou', 'dice'])
except FileNotFoundError:
    df = pd.DataFrame()

if df.empty:
    print('No results yet — run the evaluation cell first.')
else:
    print('=== Per-run Results — West Tyrol (held-out) ===')
    print(df[['model', 'seed', 'iou', 'dice']].round(4).to_string(index=False))

    agg = df.groupby('model')[['iou', 'dice']].agg(['mean', 'std']).round(4)
    agg.columns = ['iou_mean', 'iou_std', 'dice_mean', 'dice_std']
    agg = agg.reset_index()
    agg['IoU']  = agg.apply(lambda r: f"{r.iou_mean:.4f}" + (f" ± {r.iou_std:.4f}" if not np.isnan(r.iou_std) else ""), axis=1)
    agg['Dice'] = agg.apply(lambda r: f"{r.dice_mean:.4f}" + (f" ± {r.dice_std:.4f}" if not np.isnan(r.dice_std) else ""), axis=1)
    print('\n=== Aggregated ===')
    print(agg[['model', 'IoU', 'Dice']].to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

results_path = f'{REPO_DIR}/experiments/results.csv'
try:
    df = pd.read_csv(results_path).dropna(subset=['iou', 'dice'])
except FileNotFoundError:
    df = pd.DataFrame()

if df.empty:
    print('No results yet — run the evaluation cell first.')
else:
    agg = df.groupby('model')[['iou', 'dice']].agg(['mean', 'std']).round(4)
    agg.columns = ['iou_mean', 'iou_std', 'dice_mean', 'dice_std']
    agg = agg.reset_index()

    colors = ['#4C72B0', '#DD8452', '#55A868'][:len(agg)]
    fig, axes = plt.subplots(1, 2, figsize=(max(8, len(agg) * 3), 5))

    for ax, metric, label in zip(axes, ['iou', 'dice'], ['IoU', 'Dice']):
        means = agg[f'{metric}_mean']
        stds  = agg[f'{metric}_std'].fillna(0)   # NaN std → 0 (single seed)
        bars  = ax.bar(agg['model'], means, yerr=stds,
                       color=colors, capsize=6, width=0.5,
                       edgecolor='black', linewidth=0.8)
        ax.set_title(f'Validation {label}', fontsize=13)
        ax.set_ylabel(label)
        ax.set_ylim(0, 1)
        ax.axhline(y=0.70, color='red', linestyle='--', linewidth=1.2, label='70% target')
        ax.legend()
        for bar, mean in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.01,
                    f'{mean:.3f}', ha='center', va='bottom', fontsize=9)

    fig.suptitle('Aerial Building Segmentation — West Tyrol Validation', fontsize=14)
    plt.tight_layout()
    os.makedirs(f'{REPO_DIR}/experiments/figures', exist_ok=True)
    plt.savefig(f'{REPO_DIR}/experiments/figures/comparison_bar.png', dpi=120, bbox_inches='tight')
    plt.show()

## 6. Qualitative Visualisation

Metrics alone can obscure failure modes.  Here we load the best checkpoint for each architecture and display side-by-side panels:
- **Input image** — normalised RGB aerial patch
- **Ground truth** — binary building mask from INRIA annotations
- **Prediction** — thresholded sigmoid output (threshold = 0.5)

Per-tile IoU and Dice are shown below each prediction.  Five examples per model.

In [ ]:
!python src/evaluate.py --mode visualize --seed 42 --data_root {DATA_ROOT}
print('Figures saved to experiments/figures/')

In [ ]:
from IPython.display import Image, display
import glob

fig_dir = f'{REPO_DIR}/experiments/figures'
fig_paths = sorted(glob.glob(f'{fig_dir}/*_predictions.png'))

if not fig_paths:
    print('No prediction figures found — run the visualisation cell first.')
else:
    for fig_path in fig_paths:
        model_name = os.path.basename(fig_path).replace('_predictions.png', '')
        print(f'\n--- {model_name.upper()} ---')
        display(Image(filename=fig_path))

## 7. Learning Curves

The training script logs per-epoch metrics (loss, IoU, Dice, lr) to a CSV inside each checkpoint directory.  These curves show convergence speed, overfitting, and optimizer stability across seeds.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

MODELS = ['unet', 'deeplabv3plus', 'segformer']
SEEDS  = [42, 0]

available = [(m, s) for m in MODELS for s in SEEDS
             if os.path.exists(f'{CKPT_BASE}/{m}_seed{s}/metrics.csv')]

if not available:
    print('No metrics.csv files found yet. Run training first.')
else:
    fig, axes = plt.subplots(len(available), 2, figsize=(14, 4 * len(available)))
    if len(available) == 1:
        axes = [axes]

    for ax_row, (model, seed) in zip(axes, available):
        ax_iou, ax_loss = ax_row[0], ax_row[1]
        csv_path = f'{CKPT_BASE}/{model}_seed{seed}/metrics.csv'
        m = pd.read_csv(csv_path)
        ax_iou.plot(m['epoch'], m['val_iou'],   label='val',   linewidth=1.8)
        ax_iou.plot(m['epoch'], m['train_iou'], label='train', linewidth=1.2, linestyle='--')
        ax_iou.axhline(0.70, color='red', linestyle=':', linewidth=1.2, label='70% target')
        ax_iou.set_title(f'{model} seed{seed} — Val IoU')
        ax_iou.set_xlabel('Epoch'); ax_iou.set_ylabel('IoU')
        ax_iou.legend(fontsize=8); ax_iou.set_ylim(0, 1)

        ax_loss.plot(m['epoch'], m['val_loss'],   label='val')
        ax_loss.plot(m['epoch'], m['train_loss'], label='train', linestyle='--')
        ax_loss.set_title(f'{model} seed{seed} — Loss')
        ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('BCE+Dice Loss')
        ax_loss.legend(fontsize=8)

    plt.suptitle('Training Curves', fontsize=14)
    plt.tight_layout()
    os.makedirs(f'{REPO_DIR}/experiments/figures', exist_ok=True)
    plt.savefig(f'{REPO_DIR}/experiments/figures/learning_curves.png', dpi=120, bbox_inches='tight')
    plt.show()